# rw2 - solid model with embedded bars

Specimen **RW2**, Thomsen & Wallace (1995). Same wall as `rw2_stko.ipynb` (1220 x 3660, t = 102,
boundary elements 200 mm, base block 500 mm each side and 500 mm deep, axial 372.8 kN) but built with
**hexahedra** (`LadrunoBrick`, formulation `brick`) and **discrete bars** embedded in the concrete (`LadrunoEmbeddedRebar`,
perfect bond, `CorotTruss` bars). Units: **N, mm, MPa**.

| item | here | source |
|---|---|---|
| wall | 1220 x 3660 x 102, boundary elements 200 mm, split at z = 1220 (confined zone) | PEER 607 sec. 7.1.1, Fig. 7.1-7.3 |
| pedestal | 1930 x 690 x 102, one volume, elastic, E x (500 / 102) so it has the in-plane stiffness of the baseline base | PEER 607 Fig. 7.2; E baseline |
| concrete | `ASDConcrete3D`, stko `Concrete (9P)` preset, unconfined 42.8 MPa / confined 47.6 MPa; values not reported keep the calibrated STKO baseline | PEER 607 Tables 7.1-7.2 |
| boundary bars | 8 #3 (db 9.53) per end, 3 @ 51 from 19 mm, two layers 19 / 64 / 19 | PEER 607 Fig. 7.3a |
| web bars | #2 (db 6.35) @ 191, two curtains, vertical and horizontal | PEER 607 Fig. 7.3a |
| hoops | 4.76 mm wire @ 76 over the bottom 1220 mm (`lw`) | PEER 607 sec. 7.1.1, Fig. 7.3a |
| steel | `Hysteretic` per bar type: #3 434 / 505 at 2 % / 641 at 10 %; #2 and wire 448 / 519 / 586 at 8 %; pinching 0.6 / 0.8 | Table 7.1, PEER 2024/11 Table A.1; pinching baseline |
| protocol | 40 reversal points of the test record, 0.5 mm increments | test file (as `rw2_specimen`) |

Windows: runs with the Ladruno `OpenSeesMP` in **one process**. The installed apeGmsh cannot
export embedded-bar ties under partitioned MPI. Model inputs and loading history are unchanged.
Use the `C:\venv\opensees_venv\Scripts\python.exe` kernel. Evidence for every value: `context/`.

In [ ]:
import math
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from apeGmsh import apeGmsh, Results
from apeGmsh.opensees import apeSees
from apeGmsh.rebar import Cage

notebook_dir = Path(r'D:\OneDrive\Escritorio\AHORA SI\Notebooks\RW2_Wallace')
opensees_mp = Path(r'C:\OpenSees-Solvers\opensees-bin\OpenSeesMP.exe')
launcher = Path(r'C:\OpenSees-Solvers\openseesmp.bat')
nparts = 1  # installed apeGmsh cannot export embedded bars under partitioned MPI
run_dir = notebook_dir / 'rw2_solid_ladruno_embedment'
exp_file = notebook_dir / 'ExperimentalData.txt'  # m, tonf

for required in (opensees_mp, launcher, exp_file):
    if not required.is_file():
        raise FileNotFoundError(required)
run_dir.mkdir(parents=True, exist_ok=True)
print('Python:', sys.executable)
print('experiment:', exp_file)
print('run folder:', run_dir)


## inputs (N, mm, MPa)

In [ ]:
# geometry, PEER 2006/07 sec. 7.1.1, Fig. 7.2-7.3a (same as rw2_specimen)
lw = 1220.0          # wall length
hw = 3660.0          # wall height
tw = 102.0           # wall thickness (y direction, centred at y = 0)
lbe = 200.0          # boundary element width
h_conf = 1220.0      # height of the confined boundary zone (hoops @ 76), = lw
max_size = 200.0     # maximum finite element size (same as the specimen shell models)
brick = 'ssp'        # LadrunoBrick formulation: std (= stdBrick) | bbar | ssp (stabilized one point, no bending locking) | eas
bar_refine = 4       # bar mesh = max_size / bar_refine (bars 4 times finer than the concrete)
base_overhang = (1930.0 - lw) / 2    # pedestal 1.93 m long
base_depth = 690.0                   # pedestal 0.69 m deep
record_all = True    # True: every concrete node (viewer, isometric views); False: loaded nodes only (calibration runs)

# concrete, specimen values (Tables 7.1-7.2); values not reported keep the calibrated STKO baseline
ec = 31030.0                            # table 7.1-7.2
nu = 0.2
ft = 2.03                               # table 7.1
fcp_unconf, ecp_unconf = 42.8, 0.0021   # table 7.2, rw2 web (unconfined)
fcp_conf, ecp_conf = 47.6, 0.0033       # table 7.2, rw2 boundary (confined)
gt = 0.073 * fcp_unconf ** 0.18         # n/mm, fib model code 2010 (not reported)
fc0, fcr, gc = 10.0, 10.0, 36.0         # baseline (stko 01_Wallace)
pscale_t = pscale_c = 0.3               # baseline

# pedestal, elastic: baseline plate e 28198, h 500 -> same e*h over the wall thickness
base_e = 28198.0 * 500.0 / tw

# steel: fy, E, b from Table 7.1; fu, eu from PEER 2024/11 Table A.1 (as rw2_specimen)
es = 200000.0
b_h = 0.02
fy3, fu3, eu3 = 434.0, 641.0, 0.10     # #3
fy2, fu2, eu2 = 448.0, 586.0, 0.08     # #2; the hoop wire was annealed to about 448 too (sec. 7.1.2)
s2_3 = fy3 + b_h * es * (0.02 - fy3 / es)     # stress at 2 % with hardening ratio b
s2_2 = fy2 + b_h * es * (0.02 - fy2 / es)
pinch_x, pinch_y = 0.6, 0.8            # baseline (stko 01_Wallace)

# bars, fig. 7.3a
db3, db2, dbw = 9.53, 6.35, 4.76     # #3 boundary, #2 web, hoop wire
edge = 19.0                          # bar centre to face, length and thickness
be_step = 51.0                       # boundary bars 3 @ 51
web_step = 191.0                     # web bars 3 @ 191 (vertical) and @ 191 (horizontal)
hoop_step = 76.0
anchor_depth = base_depth - 50.0     # vertical bars go this deep into the pedestal (paper: 90 deg hooks)
k_perfect = 1.0e8                    # perfect-bond axial penalty of LadrunoEmbeddedRebar

# loads
p_axial = 0.07 * lw * tw * fcp_unconf   # n, 0.07 ag f'c (peer 607 sec. 6.1) = 372.8 kn
gamma_c = 2.4e-9 * 9810.0               # n/mm3, self weight

print(f'axial {p_axial / 1e3:.1f} kn | base e {base_e:.0f} mpa | gt {gt:.3f} n/mm')
print(f'steel #3 {fy3} / {s2_3:.1f} / {fu3} | #2 {fy2} / {s2_2:.1f} / {fu2} mpa')

## geometry

Same x-z layout as `rw2_stko`, built on the face y = -tw/2 and then extruded through the thickness:
one base rectangle (one physical group, one volume) and the wall rectangles, cut at z = 1220 for the
confined zone. `fragment` cuts the base top edge under the wall and, with four points, its bottom edge
at the same x, so the base meshes as one structured patch. `extrude` with layers and `recombine` turns
every quad into a column of hexahedra.

| physical group | volumes | use |
|---|---|---|
| `Wall_BE_Confined` | boundary elements, z 0 to 1220 | confined concrete |
| `Wall_Unconfined` | web, and boundary elements above 1220 | unconfined concrete |
| `Base_Block` | base block | elastic |
| `Concrete` | all volumes | host of the embedded bars |
| `Fixed` | base bottom, both base sides, exposed base top | supports |
| `Top` | wall top face | axial load, imposed displacement |

In [ ]:
g = apeGmsh(model_name='rw2_solid_ladruno_embedment', verbose=False)
g.begin()

x0, x1 = -base_overhang, lw + base_overhang
wall_x = [0.0, lbe, lw - lbe, lw]
y0 = -tw / 2                                 # extrusion face; the wall spans y -tw/2 .. tw/2

g.model.geometry.add_rectangle(x0, y0, -base_depth, x1 - x0, base_depth, plane='xz', label='base_block')
wall = []
for name, a, b in (('be_left', 0.0, lbe), ('web', lbe, lw - lbe), ('be_right', lw - lbe, lw)):
    for part, za, zb in (('low', 0.0, h_conf), ('up', h_conf, hw)):
        g.model.geometry.add_rectangle(a, y0, za, b - a, zb - za, plane='xz', label=f'{name}_{part}')
        wall.append(f'{name}_{part}')
bottom_pts = [g.model.geometry.add_point(x, y0, -base_depth) for x in wall_x]
g.model.boolean.fragment(['base_block'], wall + [(0, p) for p in bottom_pts], dim=2)

ny = math.ceil(tw / max_size - 1e-9) + 1       # nodes through the thickness
g.model.transforms.extrude(g.model.select(dim=2).tags(), 0.0, tw, 0.0, num_elements=[ny - 1], recombine=True)

tol = 1e-3
y1 = tw / 2 + tol

def vols_in(lo, hi):
    return g.model.select(dim=3).in_box(lo, hi).tags()

def faces_in(lo, hi):
    return g.model.select(dim=2).in_box(lo, hi).tags()

g.physical.add_volume(vols_in((x0 - tol, y0 - tol, -base_depth - tol), (x1 + tol, y1, tol)), name='Base_Block')
g.physical.add_volume(vols_in((-tol, y0 - tol, -tol), (lbe + tol, y1, h_conf + tol))
                      + vols_in((lw - lbe - tol, y0 - tol, -tol), (lw + tol, y1, h_conf + tol)), name='Wall_BE_Confined')
g.physical.add_volume(vols_in((lbe - tol, y0 - tol, -tol), (lw - lbe + tol, y1, h_conf + tol))
                      + vols_in((-tol, y0 - tol, h_conf - tol), (lw + tol, y1, hw + tol)), name='Wall_Unconfined')
g.physical.add_volume(g.model.select(dim=3).tags(), name='Concrete')

fixed = (faces_in((x0 - tol, y0 - tol, -base_depth - tol), (x1 + tol, y1, -base_depth + tol))   # base bottom
         + faces_in((x0 - tol, y0 - tol, -base_depth - tol), (x0 + tol, y1, tol))                # left side
         + faces_in((x1 - tol, y0 - tol, -base_depth - tol), (x1 + tol, y1, tol))                # right side
         + faces_in((x0 - tol, y0 - tol, -tol), (tol, y1, tol))                                  # exposed top, left
         + faces_in((lw - tol, y0 - tol, -tol), (x1 + tol, y1, tol)))                            # exposed top, right
g.physical.add_surface(fixed, name='Fixed')
g.physical.add_surface(faces_in((-tol, y0 - tol, hw - tol), (lw + tol, y1, hw + tol)), name='Top')

with g.loads.case('selfweight'):
    for grp in ('Wall_BE_Confined', 'Wall_Unconfined'):
        g.loads.volume(grp, force_per_volume=(0.0, 0.0, -gamma_c))

with g.loads.case('axial'):
    g.loads.surface.traction('Top', vector=(0.0, 0.0, -p_axial / (lw * tw)))

## reinforcement (Fig. 7.3a)

Bar centres (x from the left wall end, y from the wall mid-plane):

- **boundary**: 8 #3 per end, x = 19, 70, 121, 172 (and mirrored), y = +-32 (19 / 64 / 19).
- **web vertical**: #2 at 3 @ 191 centred on the wall, x = 323.5, 514.5, 705.5, 896.5, y = +-32.
- **web horizontal**: #2 @ 191 up the height, x from 19 to 1201, y = +-32 (same plane as the vertical
  bars, a truss idealisation).
- **hoops**: 4.76 mm closed rectangles around each group of 8 bars, @ 76 from z = 38 up to 1220.

Vertical bars run from `anchor_depth` inside the base to 19 mm below the top (the 90 deg hooks are not
modelled). All bars: `CorotTruss` on their own line mesh, tied to the hexahedra by `LadrunoEmbeddedRebar`
with perfect bond.

In [ ]:
yb = tw / 2 - edge                                    # 32 mm
be_x = [edge + j * be_step for j in range(4)]          # 19, 70, 121, 172
web_x = [lw / 2 + (j - 1.5) * web_step for j in range(4)]
z_bot, z_top = -anchor_depth, hw - edge

bars = []

def add_bar(points, db, material, role):
    bars.append(g.rebar.bar(points, db=db, material=material, role=role, name=f'{role}_{len(bars)}'))

for y in (-yb, yb):
    for x in be_x + [lw - x for x in be_x]:
        add_bar([(x, y, z_bot), (x, y, z_top)], db3, 'steel_3', 'vertical')
    for x in web_x:
        add_bar([(x, y, z_bot), (x, y, z_top)], db2, 'steel_2', 'vertical')
    for k in range(1, int(z_top // web_step) + 1):
        z = k * web_step
        add_bar([(edge, y, z), (lw - edge, y, z)], db2, 'steel_2', 'horizontal')

off = (db3 + dbw) / 2                                  # hoop centre line outside the bars
yh = yb + off
z = hoop_step / 2
while z < h_conf:
    for a, b in ((be_x[0] - off, be_x[-1] + off), (lw - be_x[-1] - off, lw - be_x[0] + off)):
        add_bar([(a, -yh, z), (b, -yh, z), (b, yh, z), (a, yh, z), (a, -yh, z)], dbw, 'steel_2', 'tie')
    z += hoop_step

g.rebar.place(Cage(bars=tuple(bars)), into='Concrete', coupling='embedded', perfect=k_perfect,
              emit_elements=True)

n_role = {r: sum(b.role == r for b in bars) for r in ('vertical', 'horizontal', 'tie')}
print('bars:', n_role)

## mesh

Structured hexahedra. On the x-z face each piece gets `n = ceil(length / max_size) + 1` nodes per side
(same rule as `rw2_stko`), the base is one transfinite patch with its four corners given, and the
extrusion puts `ny - 1` layers through the thickness. The bars are meshed on their own at `max_size / bar_refine` (4 times finer than the concrete).
The mesh is then split into `nparts` partitions.

In [ ]:
def n_nodes(length):
    return math.ceil(length / max_size - 1e-9) + 1

nx_be, nx_web, nx_over = n_nodes(lbe), n_nodes(lw - 2 * lbe), n_nodes(base_overhang)
nz_low, nz_up, nz_base = n_nodes(h_conf), n_nodes(hw - h_conf), n_nodes(base_depth)
print(f'element size: be {lbe / (nx_be - 1):.1f} | web {(lw - 2 * lbe) / (nx_web - 1):.1f} | '
      f'thickness {tw / (ny - 1):.1f} | confined zone {h_conf / (nz_low - 1):.1f} | '
      f'upper wall {(hw - h_conf) / (nz_up - 1):.1f} | overhang {base_overhang / (nx_over - 1):.1f} | '
      f'base depth {base_depth / (nz_base - 1):.1f} mm')

for name, nx in (('be_left', nx_be), ('web', nx_web), ('be_right', nx_be)):
    g.mesh.recipe.structured(f'{name}_low', n=(nx, nz_low), generate=False)
    g.mesh.recipe.structured(f'{name}_up', n=(nx, nz_up), generate=False)

def curves_in(lo, hi):
    return g.model.select(dim=1).in_box(lo, hi).tags()

def point_at(x, z):
    return g.model.select(dim=0).in_box((x - tol, y0 - tol, z - tol), (x + tol, y0 + tol, z + tol)).tags()[0]

spans = [(x0, 0.0, nx_over), (0.0, lbe, nx_be), (lbe, lw - lbe, nx_web), (lw - lbe, lw, nx_be), (lw, x1, nx_over)]
for z in (0.0, -base_depth):                       # base top and bottom edges, same divisions as the wall
    for a, b, n in spans:
        for c in curves_in((a - tol, y0 - tol, z - tol), (b + tol, y0 + tol, z + tol)):
            g.mesh.structured.set_transfinite_curve(c, n)
for x in (x0, x1):                                  # base sides
    for c in curves_in((x - tol, y0 - tol, -base_depth - tol), (x + tol, y0 + tol, tol)):
        g.mesh.structured.set_transfinite_curve(c, nz_base)
corners = [point_at(x0, -base_depth), point_at(x1, -base_depth), point_at(x1, 0.0), point_at(x0, 0.0)]
g.mesh.structured.set_transfinite_surface('base_block', corners=corners)
g.mesh.structured.set_recombine('base_block')
g.mesh.sizing.set_global_size(max_size / bar_refine)   # bar line mesh (the volumes are transfinite)

g.mesh.generation.generate(dim=3)
# Embedded-rebar MPI export is unsupported by the installed apeGmsh.
if nparts != 1:
    raise ValueError('This apeGmsh version requires nparts=1 for embedded bars')
fem = g.mesh.queries.get_fem_data(dim=None)

top_ids = [int(n) for n in fem.nodes.select(pg='Top').ids]
print(f'mesh : {fem.info.n_nodes} nodes | {fem.info.n_elems} elems | {nparts} partitions | types {fem.info.types}')
print(f'top nodes {len(top_ids)} | fixed nodes {len(fem.nodes.select(pg="Fixed").ids)} | '
      f'embedded ties {len(g.reinforce.reinforce_records)}')

## section cut

Set `z_cut` and run: plan view of the mesh at that height. Grey rectangles are the hexahedra cut by the
plane (dark: confined, light: unconfined); dots are bars crossing the plane (vertical bars, drawn at
their diameter); lines are bars lying in the plane within `dz` (horizontal bars blue, hoops red).
Hoops are at z = 38 + 76 k up to 1178, horizontal bars at z = 191 k.

In [ ]:
# section cut: concrete elements and bars at height z_cut
z_cut = 190.0      # mm; 190 has a hoop (38 + 2 x 76) and a horizontal bar (191) within dz
dz = 5.0           # bars lying in the plane within +-dz are drawn as lines

xyz = dict(zip(fem.nodes.ids, fem.nodes.coords))

fig, ax = plt.subplots(figsize=(14, 3))
colors = {'Wall_BE_Confined': '0.65', 'Wall_Unconfined': '0.85', 'Base_Block': '0.95'}
for grp, color in colors.items():
    for conn in fem.elements.select(pg=grp, element_type='hex8').connectivity:
        p = np.array([xyz[n] for n in conn])
        if p[:, 2].min() <= z_cut <= p[:, 2].max():
            ax.add_patch(plt.Rectangle((p[:, 0].min(), p[:, 1].min()), np.ptp(p[:, 0]), np.ptp(p[:, 1]),
                                       facecolor=color, edgecolor='0.4', lw=0.4))

bar_color = {'vertical': 'k', 'horizontal': 'tab:blue', 'tie': 'tab:red'}
for m in g.rebar.placements[0].members:
    for n1, n2 in fem.elements.select(pg=m.pg, element_type='line2').connectivity:
        a, b = np.array(xyz[n1]), np.array(xyz[n2])
        if abs(a[2] - z_cut) <= dz and abs(b[2] - z_cut) <= dz:              # bar in the cut plane
            ax.plot([a[0], b[0]], [a[1], b[1]], color=bar_color[m.role], lw=m.diameter / 3)
        elif min(a[2], b[2]) <= z_cut < max(a[2], b[2]):                     # bar crossing the cut
            t = (z_cut - a[2]) / (b[2] - a[2])
            p = a + t * (b - a)
            ax.add_patch(plt.Circle((p[0], p[1]), m.diameter / 2, color=bar_color[m.role]))

ax.set_aspect('equal')
ax.autoscale_view()
ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
ax.set_title(f'section at z = {z_cut:.0f} mm  (black: vertical bars, blue: horizontal bars, red: hoops)')
plt.show()

## mesh viewer

In [ ]:
g.mesh.viewer()

## opensees model

In [ ]:
ops = apeSees(fem)
ops.model(ndm=3, ndf=3)

def concrete(fcp, ecp):
    # Current API: reproduce the same STKO Concrete (9P) law and IMPL-EX alpha=1.
    from apeGmsh.interop.stko.translate_props import asdconcrete_9p
    from apeGmsh.opensees.material.nd import ASDConcrete3D

    law = asdconcrete_9p(dict(
        E=ec, ft=ft, fc0=fc0, fcp=fcp, fcr=fcr, ecp=ecp, Gt=gt, Gc=gc,
        **{'PScale Tension': pscale_t, 'PScale Compression': pscale_c}))
    # The current material validates damage >= 0; suppress only round-off.
    for key in ('Td', 'Cd'):
        if min(law[key]) < -1e-12:
            raise ValueError(f'Invalid negative damage in {key}')
        law[key] = tuple(max(0.0, value) for value in law[key])
    return ops.register(ASDConcrete3D(E=ec, v=nu, implex=True, **law))

conc_unconf = concrete(fcp_unconf, ecp_unconf)
conc_conf = concrete(fcp_conf, ecp_conf)
base_mat = ops.nDMaterial.ElasticIsotropic(E=base_e, nu=nu)

# bar materials: symmetric Hysteretic (yield, stress at 2 %, fu at eu), by the names used in g.rebar.bar
def hysteretic(fy, e_s, s2, fu, eu, name):
    ey = fy / e_s
    assert ey < 0.02 < eu and fy < s2 < fu, (fy, s2, fu, eu)
    return ops.uniaxialMaterial.Hysteretic(
        s1p=fy, e1p=ey, s2p=s2, e2p=0.02, s3p=fu, e3p=eu,
        s1n=-fy, e1n=-ey, s2n=-s2, e2n=-0.02, s3n=-fu, e3n=-eu,
        pinch_x=pinch_x, pinch_y=pinch_y, damage1=0.0, damage2=0.0, name=name)

hysteretic(fy3, es, s2_3, fu3, eu3, 'steel_3')     # #3 boundary bars
hysteretic(fy2, es, s2_2, fu2, eu2, 'steel_2')     # #2 web bars and hoop wire

ops.element.LadrunoBrick(pg='Wall_Unconfined', material=conc_unconf, formulation=brick)
ops.element.LadrunoBrick(pg='Wall_BE_Confined', material=conc_conf, formulation=brick)
ops.element.LadrunoBrick(pg='Base_Block', material=base_mat, formulation=brick)

ops.fix(pg='Fixed', dofs=(1, 1, 1))

## loading protocol (40 reversal points of the test record, same as rw2_specimen)

In [ ]:
peaks = [2.34, -2.61, 2.45, -2.38, 5.91, -6.5, 6.11, -6.52, 13.01, -13.7,
         13.84, -13.85, 20.15, -21.22, 20.56, -21.12, 28.52, -29.41, 28.75, -29.71,
         45.52, -46.01, 45.75, -46.12, 29.49, -29.84, 29.34, -29.99, 45.61, -46.12,
         45.27, -46.0, 62.12, -62.09, 62.45, -62.32, 79.85, -79.23, 80.46, -79.57]   # mm
disp_values = [0.0] + peaks
travel = np.cumsum(np.abs(np.diff(disp_values)))       # mm travelled up to each peak
time_values = [0.0] + [float(t) for t in travel / travel[-1]]   # pseudo-time proportional to travel
d_step = 0.5                                      # mm, displacement increment
n_cyclic = math.ceil(travel[-1] / d_step)
# hold the last peak a little past t = 1: the accumulated pseudo-time of the last increment can land a hair
# beyond 1.0, where the Path series returns 0 (a jump back to zero displacement)
disp_values.append(peaks[-1])
time_values.append(1.0 + 1.0 / n_cyclic)
print(f'{len(peaks)} peaks, {min(peaks):.2f} / {max(peaks):.2f} mm | travel {travel[-1]:.1f} mm | '
      f'{n_cyclic} increments of {travel[-1] / n_cyclic:.3f} mm')

## stages and deck

1. `gravity`: self weight + 372.8 kN axial as a traction on the top face, 10 increments, then held.
2. `cyclic`: `sp` ux on every top node following the path.

Windows single-process export: `RCM` numberer and `UmfPack` direct solver (the installed
bridge rejects serial `Mumps`). One `.ladruno` recorder (displacement + reaction of
every node) in the cyclic stage. Base shear = sum of the x reactions of the top nodes.

`record_all = True` records every concrete node (results viewer, isometric views); `False` records only
the loaded nodes (small files, enough for the base shear curve; used in the calibration runs).

In [ ]:
rec = ops.recorder.Ladruno(file='rw2_solid_ladruno_embedment.ladruno', nodal_responses=('displacement', 'reactionForce'),
                           nodes_pg='Concrete' if record_all else 'Top')

n_gravity = 10
dlam_gravity = 1.0 / n_gravity

with ops.stage(name='gravity') as s:
    with s.pattern(series=ops.timeSeries.Linear()) as p:
        p.from_model('selfweight')
        p.from_model('axial')
    s.analysis(test=ops.test.NormDispIncr(tol=1e-4, max_iter=50),
               algorithm=ops.algorithm.KrylovNewton(max_dim=10),
               integrator=ops.integrator.LoadControl(dlam=dlam_gravity),
               constraints=ops.constraints.Transformation(),
               numberer=ops.numberer.RCM(),
               system=ops.system.UmfPack(),
               analysis=ops.analysis.Static())
    s.run(n_increments=n_gravity, dt=dlam_gravity)

dlam_cyclic = 1.0 / n_cyclic

with ops.stage(name='cyclic') as s:
    with s.pattern(series=ops.timeSeries.Path(values=disp_values, time=time_values)) as p:
        p.sp(pg='Top', dof=1, value=1.0)
    s.recorder(rec)
    s.analysis(test=ops.test.NormDispIncr(tol=1e-4, max_iter=100),
               algorithm=ops.algorithm.KrylovNewton(max_dim=10),
               integrator=ops.integrator.LoadControl(dlam=dlam_cyclic),
               constraints=ops.constraints.Transformation(),
               numberer=ops.numberer.RCM(),
               system=ops.system.UmfPack(),
               analysis=ops.analysis.Static())
    # on a failed increment escalate ModifiedNewton(initial) -> KrylovNewton -> NewtonLineSearch
    s.run(n_increments=n_cyclic, dt=dlam_cyclic, strategy=ops.strategy.profile('penalty-stiff'))

ops.tcl(str(run_dir / 'rw2_solid_ladruno_embedment.tcl'))
g.end()
print('deck:', run_dir / 'rw2_solid_ladruno_embedment.tcl')

## run (Windows OpenSeesMP, one process)

The Windows launcher initializes the solver environment. On failure or interruption, only this run
and its child processes are stopped.


In [ ]:
deck_file = run_dir / 'rw2_solid_ladruno_embedment.tcl'
if not deck_file.is_file():
    raise FileNotFoundError(f'Generate the deck first: {deck_file}')
if nparts != 1:
    raise ValueError('This apeGmsh version requires nparts=1 for embedded bars')

for f in run_dir.glob('rw2_solid_ladruno_embedment*.ladruno'):   # clean HDF5 recorders on rerun
    f.unlink()

log_file = run_dir / 'rw2_solid_ladruno_embedment_run.log'
failed = False
# With no second argument, this Windows launcher runs OpenSeesMP in one process.
# It also initializes Intel MPI, Tcl and the solver DLL search paths.
with log_file.open('w', encoding='utf-8') as log:
    proc = subprocess.Popen(
        [os.environ.get('COMSPEC', 'cmd.exe'), '/d', '/c', str(launcher), str(deck_file)],
        cwd=run_dir, stdout=log, stderr=subprocess.STDOUT,
        creationflags=subprocess.CREATE_NO_WINDOW)
    try:
        while proc.poll() is None:
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                pass
            text = log_file.read_text(encoding='utf-8', errors='replace')
            if any(token in text for token in ('FAILED', 'ERROR', 'FATAL', 'while executing')):
                failed = True
                break
    finally:
        if proc.poll() is None:   # this run only, including children; also handles kernel interrupts
            subprocess.run(['taskkill', '/PID', str(proc.pid), '/T', '/F'],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                           creationflags=subprocess.CREATE_NO_WINDOW, check=False)
            proc.wait(timeout=30)

text = log_file.read_text(encoding='utf-8', errors='replace')
failed = failed or any(token in text for token in ('FAILED', 'ERROR', 'FATAL', 'while executing'))
print('return code:', proc.returncode, '| failed:', failed)
print([line for line in text.splitlines() if 'APEGMSH_PROGRESS' in line][-1:])
assert not failed and proc.returncode == 0, f'opensees run failed, see {log_file}'
assert f'APEGMSH_PROGRESS i={n_cyclic} n={n_cyclic} ' in text, f'Incomplete cyclic analysis, see {log_file}'


## results vs experiment

| source | file units | conversion |
|---|---|---|
| model (`.ladruno`, one file per rank, merged by `Results.from_ladruno`) | disp mm, reaction N | sum of top-node x reactions / 1000 |
| experiment (`ExperimentalData.txt`) | disp **m**, force **tonf** | disp x 1000, force x 9.80665 |

In [ ]:
import h5py


def _has_nodes(path):
    with h5py.File(path, 'r') as f:
        return any(isinstance(g, h5py.Group) and 'MODEL/NODES' in g for g in f.values())


name = 'rw2_solid_ladruno_embedment'
# a rank that owns none of the recorded nodes writes an empty file: read only the files with data
parts = [str(p) for p in sorted(run_dir.glob(f'{name}*.ladruno')) if _has_nodes(p)]
results = Results.from_ladruno(parts, merge_partitions=True)

disp = results.nodes.get(ids=top_ids, component='displacement_x').values[:, 0]                    # mm
shear = results.nodes.get(ids=top_ids, component='reaction_force_x').values.sum(axis=1) / 1000.0  # kn
n = min(len(disp), len(shear))
disp, shear = disp[:n], shear[:n]

exp = np.loadtxt(exp_file)
exp_disp, exp_shear = exp[:, 0] * 1000.0, exp[:, 1] * 9.80665

for name, d, v in (('solid', disp, shear), ('experiment', exp_disp, exp_shear)):
    print(f'{name:10s}: {len(d):5d} pts | disp {d.min():7.1f} / {d.max():6.1f} mm | shear {v.min():7.1f} / {v.max():6.1f} kn')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(exp_disp, exp_shear, 'k--', lw=1.0, label='experiment (Thomsen & Wallace 1995)')
ax.plot(disp, shear, lw=1.0, label=f'solid, embedded bars (max_size {max_size:.0f})')
ax.set_xlabel('top displacement (mm)')
ax.set_ylabel('base shear (kN)')
ax.set_title('RW2 solid - base shear vs top displacement')
ax.grid(True)
ax.legend()
plt.show()

## results viewer

In [ ]:
results.viewer()

## results only (run from here)

Stand-alone: imports, reads this notebook's deck and `.ladruno` files from the last run and plots against the experiment. No need to run the cells above.

In [ ]:
# results only: run this cell by itself (fresh kernel is fine): it reads the deck and the .ladruno of the last run
import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from apeGmsh import Results
import h5py


def _has_nodes(path):
    with h5py.File(path, 'r') as f:
        return any(isinstance(g, h5py.Group) and 'MODEL/NODES' in g for g in f.values())


name = 'rw2_solid_ladruno_embedment'
notebook_dir = Path(r'D:\OneDrive\Escritorio\AHORA SI\Notebooks\RW2_Wallace')
run_dir = notebook_dir / name
exp_file = notebook_dir / 'ExperimentalData.txt'   # m, tonf

deck = (run_dir / f'{name}.tcl').read_text(encoding='utf-8')
top_ids = sorted({int(n) for n in re.findall(r'^\s*sp (\d+) 1 ', deck, re.M)})   # nodes with the imposed displacement

# a rank that owns none of the recorded nodes writes an empty file: read only the files with data
parts = [str(p) for p in sorted(run_dir.glob(f'{name}*.ladruno')) if _has_nodes(p)]
results = Results.from_ladruno(parts, merge_partitions=True)

disp = results.nodes.get(ids=top_ids, component='displacement_x').values[:, 0]                    # mm
shear = results.nodes.get(ids=top_ids, component='reaction_force_x').values.sum(axis=1) / 1000.0  # kn
n = min(len(disp), len(shear))
disp, shear = disp[:n], shear[:n]

exp = np.loadtxt(exp_file)
exp_disp, exp_shear = exp[:, 0] * 1000.0, exp[:, 1] * 9.80665   # m -> mm, tonf -> kn

for label, d, v in ((name, disp, shear), ('experiment', exp_disp, exp_shear)):
    print(f'{label:20s}: {len(d):5d} pts | disp {d.min():7.1f} / {d.max():6.1f} mm | shear {v.min():8.1f} / {v.max():7.1f} kn')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(exp_disp, exp_shear, 'k--', lw=1.0, label='experiment')
ax.plot(disp, shear, lw=1.0, label=name)
ax.set_xlabel('top displacement (mm)')
ax.set_ylabel('base shear (kN)')
ax.set_title('RW2 - base shear vs top displacement')
ax.grid(True)
ax.legend()
plt.show()